# DO NOT CHANGE
- this notebooks is for creating the charts for the final submission report
- is applies Nash Product alpha=0.5 on org_id=1, in January

In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, plot_stacked_gain_loss_sortable, tf_to_int, tf_to_hourly, tf_to_daily
from modules.visualisations import plot_profile_by_category,plot_distribution_comparison,plot_sorted_mps_comparison

from plotly.io import to_html
from IPython.display import display, HTML
import cvxpy as cp

from modules.optimization_algorithms.NashProductOptAlgo import NashProductOptAlgo
from modules.optimization_algorithms.XNashProductOptAlgo import XNashProductOptAlgo
from modules.optimization_algorithms.EqualWaterfillingOptAlgo import EqualWaterfillingOptAlgo



from modules.fairness_metric_calculator import TheilIndexCalculator, GiniCoeffCalculator, JainIndexCalculator, AtkinsonIndexCalculator

In [ ]:
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

## Params

In [ ]:
# # PARMS
# changeable
org_id = 1
start_time = datetime(2025,1, 7)
end_time = datetime(2025, 1, 28)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

## Load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")
raw_eeg.head()

In [ ]:
eeg_selected_feat = raw_eeg[["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del raw_eeg
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop("metering_point_id", axis=1).describe())
eeg_selected_feat.head()

In [ ]:
work = eeg_selected_feat[
    (eeg_selected_feat["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (eeg_selected_feat["time"] < pd.Timestamp(end_time, tz='UTC'))
].drop_duplicates().copy()
del eeg_selected_feat
# cons_gen: Consumed Generation, how much of the generated electricity was consumed within the EEG
work["cons_gen"] = work["wt_meas_gen"] - work["wt_surp_gen"]
work.sum(numeric_only=True)

In [ ]:
work[work["energy_direction"]=="G"].head()

## eda

In [ ]:
def plot_sorted_mps(data:pd.DataFrame, feature:str, show=False) -> None:

    df = data.groupby(by="metering_point_id").sum(numeric_only=True)[feature]

    df = df.sort_values(ascending=False)
    x = np.arange(1, len(df) + 1)
    y = df.values
    mp_ids = df.index  # for hover

    # uniform colours
    bar_color = 'lightblue'
    hist_color = 'lightblue'
    box_color = 'lightblue'
    border_color = 'black'

    # subplots: 3 rows, 1 column
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.1,
        subplot_titles=(f"Sorted {df.name}-Values", f"Histogramm [{df.name}]", f"Boxplot [{df.name}]")
    )

    # bar plot on top
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        hovertext=mp_ids,
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
        name="Sorted values",
        marker=dict(color=bar_color, line=dict(color=border_color, width=1))
    ), row=1, col=1)

    # histogram in the middle
    fig.add_trace(go.Histogram(
        x=y,
        nbinsx=100,  # adjust number of bins
        name="Histogram",
        marker=dict(color=hist_color, line=dict(color=border_color, width=1)),
    ), row=2, col=1)

    # horizontal boxplot at the bottom
    fig.add_trace(go.Box(
        x=y,
        orientation='h',
        boxpoints='outliers',  # show outliers
        marker=dict(color=border_color),
        line=dict(color=border_color),
        name="Boxplot"
    ), row=3, col=1)

    # layout adjustments
    fig.update_layout(
        title_text=f"{df.name} summed up on single metering points from {start_time.date()} - {end_time.date()} ({len(data.time.unique())} timestamps), {len(df)} mp ids, ",
        template="plotly_white",
        showlegend=False,
        height=900
    )

    # axis titles
    fig.update_xaxes(title_text="sorted Index", row=1, col=1)
    fig.update_yaxes(title_text=df.name, row=1, col=1)
    fig.update_xaxes(title_text=f"{df.name}", row=2, col=1)
    fig.update_yaxes(title_text="count", row=2, col=1)
    fig.update_xaxes(title_text=f"{df.name}", row=3, col=1)
    fig.update_yaxes(title_text="", row=3, col=1)
    
    if show:
        fig.show()
    return fig

In [ ]:
feature = "cons_gen" # "wt_meas_gen" # "comm_cov"
energy_direction_filter = "G" # "C"

In [ ]:
plot_sorted_mps(data=work[work["energy_direction"]==energy_direction_filter], feature=feature)

In [ ]:
temp_gens = work[work["energy_direction"]==energy_direction_filter].groupby("metering_point_id").sum(numeric_only=True).drop(columns=["wt_meas_cons", "comm_pot", "comm_cov"])
temp_gens["cons_gen"] = temp_gens["wt_meas_gen"] - temp_gens["wt_surp_gen"]
temp_gens["cons_gen_to_wt_meas_gen_ratio"] = temp_gens["cons_gen"] / temp_gens["wt_meas_gen"]
temp_gens

In [ ]:
# temp_gens = work[(work["energy_direction"]==energy_direction_filter) & (work["time"].isin(time_with_surplus.time))].groupby("mp_id").sum(numeric_only=True).drop(columns=["wt_meas_cons", "comm_pot", "comm_cov"])
# temp_gens["cons_gen"] = temp_gens["wt_meas_gen"] - temp_gens["wt_surp_gen"]
# temp_gens["cons_gen_to_wt_meas_gen_ratio"] = temp_gens["cons_gen"] / temp_gens["wt_meas_gen"]
# temp_gens

In [ ]:
# random_value = np.random.choice(time_with_surplus["time"].unique())

# temp_gens = (
#     work[
#         (work["energy_direction"] == energy_direction_filter)
#         & (work["time"] == random_value)
#     ]
#     .groupby(["time", "mp_id"])
#     .sum(numeric_only=True)
#     .drop(columns=["wt_meas_cons", "comm_pot", "comm_cov"])
#     .reset_index()
# )
# temp_gens["cons_gen"] = temp_gens["wt_meas_gen"] - temp_gens["wt_surp_gen"]
# temp_gens["cons_gen_to_wt_meas_gen_ratio"] = (
#     temp_gens["cons_gen"] / temp_gens["wt_meas_gen"]
# )
# temp_gens

In [ ]:
# plot_sorted_mps(data=temp_gens, feature="cons_gen")

### Waterfilling Opt for finding optimal pfs

In [ ]:
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["metering_point_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_wt_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_wt_meas_gen", "wt_surp_gen":"sum_wt_surp_gen", "cons_gen":"sum_cons_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")
agg_on_time.head()

In [ ]:
time_with_deficit = agg_on_time[agg_on_time["sum_wt_surp_gen"] <= 0]
time_with_surplus = agg_on_time[agg_on_time["sum_wt_surp_gen"] > 0]

print(f"{len(time_with_deficit)}/{len(agg_on_time)} ({(len(time_with_deficit)/len(agg_on_time))*100:.4}%) timestamps has deficit. only for consumers during deficit a pf is optimized")
print(f"{len(time_with_surplus)}/{len(agg_on_time)} ({(len(time_with_surplus)/len(agg_on_time))*100:.4}%) timestamps has surplus. only for generators during surplus a pf is optimized")
time_with_deficit.head()

#### Acutal optimization calculation

In [ ]:
time_for_pf_opt = time_with_deficit if energy_direction_filter == "C" else time_with_surplus
feat_for_r0_pf_opt = "wt_meas_cons" if energy_direction_filter == "C" else "wt_meas_gen"
feat_for_R0_pf_opt = "sum_wt_meas_gen" if energy_direction_filter == "C" else "sum_wt_meas_cons"
feat_as_opt_target = "comm_cov" if energy_direction_filter == "C" else "cons_gen"
feat_for_aopt_pf_opt = f"{feat_as_opt_target}_opt"

In [ ]:
solver = cp.SCS # cp.SCS # cp.ECOS
algo = NashProductOptAlgo(alpha=0.5, solver=solver) 
# algo = EqualWaterfillingOptAlgo () # 0.50, 0, 1, 1.5, 

In [ ]:
work_deficit = work # pd.merge(left=work[work["energy_direction"] == energy_direction_filter], right=time_for_pf_opt, on="time", how="inner")
tf_schedule_raw = algo.calculate_pfs(work_deficit)
# tf_schedule_raw["pf"] = tf_schedule_raw["pf"].replace([np.inf, -np.inf], 100)



In [ ]:
# tf_schedule_raw.to_csv("../../local_data/raw_nashProductTFS_a050_sSCS.csv", index=False)
tf_schedule_raw = pd.read_csv("../../local_data/raw_nashProductTFS_a050_sSCS.csv")
tf_schedule_raw["time"] = pd.to_datetime(tf_schedule_raw["time"], utc=True)

In [ ]:
tf_schedule_int = tf_to_int(tf_schedule_raw)
tf_schedule_int.head()

### Apply pf schedule to energy data

In [ ]:


tf_schedule_hourly = tf_to_hourly(tf_schedule_int)

In [ ]:


tf_schedule_daily = tf_to_daily(tf_schedule_int)

In [ ]:
applied_15min_int_pfs = apply_pf_schedule_to_mps(work, tf_schedule_int)
applied_hourly_int_pfs = apply_pf_schedule_to_mps(work, tf_schedule_hourly)
applied_daily_int_pfs = apply_pf_schedule_to_mps(work, tf_schedule_daily)

In [ ]:
def print_opt_energy_data(applied_pfs: pd.DataFrame) -> None:
    """
    Output of optimized values after participation factor schedule apply.
    Dynamically prints all relevant energy features with sums and comparisons.
    """

    # baseline & optimized Restnetzbezug
    baseline_restnetzbezug = (
        applied_pfs["wt_meas_cons"].sum() - applied_pfs["comm_cov"].sum()
    )
    opt_restnetzbezug = (
        applied_pfs["opt_wt_meas_cons"].sum() - applied_pfs["comm_cov"].sum()
    )

    print("\nOptimized Sums:")

    # Features to report: dynamically handle all columns starting with 'opt_'
    cons_feature_map = {
        "opt_wt_meas_cons": ("opt_meas_cons (c*)", "wt_meas_cons"),
        "opt_comm_cov": ("opt_comm_cov (cc*)", "comm_cov"),
        "opt_comm_pot": ("opt_comm_pot (cp*)", "comm_pot"),
    }

    gen_feature_map = {
        "opt_wt_meas_gen": ("opt_wt_meas_gen (g*)", "wt_meas_gen"),
        "opt_wt_surp_gen": ("opt_wt_surp_gen (s*)", "wt_surp_gen"),
        "opt_cons_gen": ("opt_cons_gen (cg*)", "cons_gen"),
    }

    def _print_feature_changes(feature_map:dict, category:str) -> None:
        """Prints the changes from baseline energy flow features to optimized energy flow features."""
        print(f"\t{category}")
        for opt_col, (label, base_col) in feature_map.items():
            opt_sum = applied_pfs[opt_col].sum()
            if base_col:
                base_sum = applied_pfs[base_col].sum()
                diff = base_sum - opt_sum
                print(
                    f"\t\t{label}: {opt_sum:.3f} [base {base_col}: {base_sum:.3f}, difference: {diff:.3f}]"
                )
            else:
                print(f"\t\t{label}: {opt_sum:.3f}")

    _print_feature_changes(cons_feature_map, "Consumers:")
    

    print(f"\t-> opt_Restnetzbezug (c* - cc*): {opt_restnetzbezug:.3f}")
    print(f"\t-> real Restnetzbezug: {baseline_restnetzbezug:.3f}")
    print(
        f"\t-> real Restnetzbezug + opt_comm_cov = {baseline_restnetzbezug:.3f} + {applied_pfs['opt_comm_cov'].sum():.3f} = {applied_pfs['wt_meas_cons'].sum():.3f}"
    )

    _print_feature_changes(gen_feature_map, "Generators:")

In [ ]:
print_opt_energy_data(applied_15min_int_pfs)

In [ ]:
print_opt_energy_data(applied_hourly_int_pfs)

In [ ]:
print_opt_energy_data(applied_daily_int_pfs)

## Evaluate overall energy flow Balance

In [ ]:
move = applied_daily_int_pfs[(applied_daily_int_pfs["energy_direction"]=="C") & ((applied_daily_int_pfs["comm_cov"] - applied_daily_int_pfs["opt_comm_cov"]).abs() <= 5e-4) & (applied_daily_int_pfs["pf"]!=1)]

move[["time", "metering_point_id", "wt_meas_cons", "comm_cov", "opt_wt_meas_cons", "opt_comm_cov", "pf"]]

In [ ]:

print(move[["time", "metering_point_id", "wt_meas_cons", "comm_cov", "opt_wt_meas_cons", "opt_comm_cov", "pf"]].head())

In [ ]:
move = applied_15min_int_pfs[(applied_15min_int_pfs["energy_direction"]=="G") & ((applied_15min_int_pfs["cons_gen"] - applied_15min_int_pfs["opt_cons_gen"]).abs() <= 5e-4) & (applied_15min_int_pfs["pf"]!=1)]

move[["time", "metering_point_id", "wt_meas_gen", "cons_gen", "opt_wt_meas_gen", "opt_cons_gen", "pf"]]

#### Summed up for 15min

In [ ]:
temp_applied_pfs_ed = applied_15min_int_pfs # applied_hourly_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_hourly_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_hourly_int_pfs # applied_daily_int_pfs # 

In [ ]:
check_full_calc_via_time = temp_applied_pfs_ed.groupby(by="time").sum()[
    [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen"
    ]
].add_prefix("sum_")
check_full_calc_via_time

#### Summed up for single MP over whole time horizon

In [ ]:
check_full_calcc_via_mpid = temp_applied_pfs_ed.groupby(by="metering_point_id").sum(numeric_only=True)[
    [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
].add_prefix("sum_")
check_full_calcc_via_mpid.sort_values(by="sum_comm_cov_delta", ascending=False)

#### Summed up for single MP over single timestamp

### Result evaluation

In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min
random_timestamp_from_tf_opt = np.random.choice(time_with_deficit.time)
print(
    "Random timestamp from time where pf opt got applied:", random_timestamp_from_tf_opt
)
# random_timestamp_from_tf_opt = datetime.fromisoformat(
#     "2025-01-24 04:45:00+00:00"
# )
# 2) Filter dt by this value
single_time_filtered = temp_applied_pfs_ed[temp_applied_pfs_ed["time"] == random_timestamp_from_tf_opt]

check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"]=="C"]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
print(check_full_calc_via_single_timestamp.sum(numeric_only=True))
check_full_calc_via_single_timestamp.sort_values(by="sum_comm_cov_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "sum_comm_cov", "sum_opt_comm_cov")

In [ ]:
print(single_time_filtered[single_time_filtered["metering_point_id"].isin([12581, 12622])][["time", "organization_id", "metering_point_id", "wt_meas_cons", "comm_cov", "pf", "opt_wt_meas_cons", "opt_comm_cov"]].round(3))

In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min
random_timestamp_from_tf_opt = np.random.choice(time_with_deficit.time.dt.date.unique())
print(
    "Random timestamp from time where pf opt got applied:", random_timestamp_from_tf_opt
)
# 2) Filter dt by this value
single_time_filtered = temp_applied_pfs_ed[
    (temp_applied_pfs_ed["time"].dt.date == random_timestamp_from_tf_opt)
]



In [ ]:
check_full_calc_via_single_timestamp = (
    single_time_filtered
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
# print(check_full_calc_via_single_timestamp.sum(numeric_only=True))

# Consumers
check_full_calc_via_single_timestamp.sort_values(by="sum_comm_cov_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "sum_comm_cov", "sum_opt_comm_cov")

# Generators
# check_full_calc_via_single_timestamp.sort_values(by="sum_cons_gen_delta", ascending=False)
# plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "sum_cons_gen", "sum_opt_cons_gen")

In [ ]:
single_time_filtered.time.min()

In [ ]:
single_time_filtered.time.max()

In [ ]:
temp_applied_pfs_ed = applied_daily_int_pfs # applied_15min_int_pfs # applied_hourly_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_hourly_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_daily_int_pfs # applied_hourly_int_pfs # applied_15min_int_pfs # applied_hourly_int_pfs # applied_daily_int_pfs # 

In [ ]:
single_time_filtered = temp_applied_pfs_ed
check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"]=="C"]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
# print(check_full_calc_via_single_timestamp.sum(numeric_only=True))
check_full_calc_via_single_timestamp.sort_values(by="sum_comm_cov_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "sum_comm_cov", "sum_opt_comm_cov")

# Generators
#check_full_calc_via_single_timestamp.sort_values(by="sum_cons_gen_delta", ascending=False)
#plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "sum_cons_gen", "sum_opt_cons_gen")

In [ ]:
print(single_time_filtered[single_time_filtered["metering_point_id"].isin([12581, 12622])].groupby("metering_point_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "opt_wt_meas_cons", "opt_comm_cov"]].round(3))

In [ ]:
keys = ["time", "organization_id", "metering_point_id", "energy_direction"]

df_15m = applied_15min_int_pfs.rename(
    columns={c: f"{c}_15min" for c in applied_15min_int_pfs.columns if c not in keys}
)

df_1h = applied_hourly_int_pfs.rename(
    columns={c: f"{c}_1h" for c in applied_hourly_int_pfs.columns if c not in keys}
)

df_1d = applied_daily_int_pfs.rename(
    columns={c: f"{c}_1d" for c in applied_daily_int_pfs.columns if c not in keys}
)

tfs_comparison_raw = (
    df_15m
    .merge(df_1h, on=keys)
    .merge(df_1d, on=keys)
)

tfs_comparison_raw = tfs_comparison_raw.drop(
    columns=[c for c in tfs_comparison_raw.columns if c.startswith("sum_")]
)

tfs_comparison = (
    tfs_comparison_raw[tfs_comparison_raw["energy_direction"]=="C"]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)
).add_prefix("sum_")
print(tfs_comparison.columns)
tfs_comparison.head()

In [ ]:
metrics = {
    "Gini": GiniCoeffCalculator(),
    "Theil": TheilIndexCalculator(),
    "Jain": JainIndexCalculator(),
    "Atkinson e=0.5": AtkinsonIndexCalculator(0.5),
    "Atkinson e=1.0": AtkinsonIndexCalculator(1),
    "Atkinson e=1.5": AtkinsonIndexCalculator(1.5),
    "Atkinson e=2.0": AtkinsonIndexCalculator(2),
    "Atkinson e=2.5": AtkinsonIndexCalculator(2.5),
}

timeframes = ["15min", "1h", "1d"]

lines = []
lines.append("| Metric | Baseline | Opt 15min | Opt 1h | Opt 1d |")
lines.append("|--------|------|-----------|--------|--------|")

for metric_name, metric in metrics.items():
    comm = round(
        metric.calculate_evaluation_metric(
            tfs_comparison["sum_comm_cov_15min"]
        ),
        4,
    )

    opt_values = []
    for tf in timeframes:
        val = metric.calculate_evaluation_metric(
            tfs_comparison[f"sum_opt_comm_cov_{tf}"]
        )
        opt_values.append(round(val, 4))

    lines.append(
        f"| {metric_name} | {comm} | {opt_values[0]} | {opt_values[1]} | {opt_values[2]} |"
    )

markdown_table = "\n".join(lines)
print(markdown_table)

In [ ]:
print(
    tfs_comparison.reset_index()[
        [
            "metering_point_id",
            "sum_comm_cov_15min",
            "sum_opt_comm_cov_15min",
            "sum_opt_comm_cov_1h",
            "sum_opt_comm_cov_1d",
        ]
    ]
    .round(3)
    .sort_values("sum_comm_cov_15min")
    .head()
)

In [ ]:
temp_tfs_comparison = (
    tfs_comparison_raw
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)
).add_prefix("sum_")

temp_tfs_comparison.sum()[["sum_comm_cov_15min", "sum_opt_comm_cov_15min", "sum_opt_comm_cov_1h", "sum_opt_comm_cov_1d"]]

In [ ]:
# TODO cahrt with 2 comparing histgrams, comparing between cc and cc*

In [ ]:
# OPTIMIZATION CHANGES on while time horizon
plot_stacked_gain_loss_sortable(sums_on_cons_mps, "comm_cov", "opt_comm_cov")

#### Explanation of 15min-Opt vs "Full Horizon"-Opt

In [ ]:
obj_ids_to_filter = [238, 134, 162, 25]

temp = applied_pfs[applied_pfs["energy_direction"] == "C"]
#temp = temp[temp["mp_id"].isin(obj_ids_to_filter)]
plot_profile_by_category(temp, energy_col_name='wt_meas_cons', agg_func_str='median', hue_col="metering_point_id", logo=logo)

- chart nash -> no change
perspective: single perspective: no change in comm_cov, but tf was set

-> perspective eeg sums -> no change in comm_cov, but transfer of cons


- pipeline sketch
- pipeline draw io